In [242]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier 


pd.set_option('display.max_columns', None)   # 모든 컬럼 표시
pd.set_option('display.width', None)         # 줄바꿈 없이 전체 폭 사용
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 생략 안 함print(df)

df = pd.read_csv('data/reviews_joined_all_matched.csv')
df.head()

C:\Users\Playdata\AppData\Local\Temp\ipykernel_29920\3874034859.py:20: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/reviews_joined_all_matched.csv')


,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,developer_response,timestamp_dev_responded,primarily_steam_deck,appid_1,game_name,genre
0,2139460,199685023,76561198220582271,23,1,15240.0,0.0,15240,NaN,1.748034e+09,turkish,oyunu kurduk eyv oynadık vs zaten türk ü bırak oyuncu bulmak çok zor sadece rus kekolar var bol bol hadi bunları geçtik diyelim oynadık durduk bir güncelleme geldi neymiş yeni senaryo imiş tamam eyv süper güncelleme geliyor bi baktım 70 gb dedik oyun bayağı değişiyor gelişiyor neyse yaptık güncellemeyi baktım bi tek senaryo gelmiş değişen birşey yok tamam dedik eyv. geçen bir kere daha gireyim dedim yeni senaryo gelmiş bi baktım şimdi 64 gb lık bir güncelleme daha .Olum siz manyak mısınız zaten tek tük oyuncu var oynayan birde her senaryoda 70gb güncelleme mi olur mk. mal mısınız sildim sokarım oyununuza. Undawn a devam mk,1752389967,1752389967,False,17,2,0.712445,1,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"
1,2139460,199684668,76561198401253542,0,1,6249.0,0.0,834,NaN,1.755746e+09,spanish,me esta fasinando,1752389691,1752389691,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"
2,2139460,199683985,76561199435851437,0,1,20522.0,0.0,12327,NaN,1.755124e+09,english,truly an amazing game,1752389204,1752389204,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"
3,2139460,199683578,76561198977144059,83,30,22726.0,0.0,4090,NaN,1.755366e+09,english,amazing free to play .,1752388914,1752388914,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"
4,2139460,199683575,76561199109403538,0,1,2747.0,0.0,2389,NaN,1.752985e+09,schinese,好,1752388911,1752388911,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']"


In [243]:
df.describe()

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,timestamp_created,timestamp_updated,votes_up,votes_funny,weighted_vote_score,comment_count,timestamp_dev_responded,appid_1
count,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030645e+06,1.030645e+06,1.030656e+06,19544.000000,1.030645e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,3.506000e+03,1.030656e+06
mean,1.246007e+06,2.078202e+08,7.656120e+16,5.589029e+01,9.407134e+00,1.421312e+04,4.750142e+02,1.179464e+04,1883.487055,1.762456e+09,1.760941e+09,1.761054e+09,6.485481e-01,1.244169e-01,5.027320e-01,4.645294e-02,1.759162e+09,1.246007e+06
std,8.606580e+05,4.897537e+06,6.086190e+08,1.973416e+02,6.927022e+01,3.357524e+04,1.155320e+03,3.145326e+04,5554.090679,1.173887e+07,4.747926e+06,4.725906e+06,1.345458e+01,3.082606e+00,2.275669e-02,6.260744e-01,4.410209e+06,8.606580e+05
min,4.400000e+02,1.994023e+08,7.656120e+16,0.000000e+00,1.000000e+00,5.000000e+00,0.000000e+00,5.000000e+00,1.000000,1.345532e+09,1.752096e+09,1.752096e+09,0.000000e+00,0.000000e+00,1.352486e-01,0.000000e+00,1.752181e+09,4.400000e+02
25%,5.268700e+05,2.032641e+08,7.656120e+16,0.000000e+00,1.000000e+00,1.440000e+03,0.000000e+00,8.080000e+02,46.000000,1.761593e+09,1.756657e+09,1.756828e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.755116e+09,5.268700e+05
50%,1.172470e+06,2.081491e+08,7.656120e+16,0.000000e+00,3.000000e+00,4.408000e+03,0.000000e+00,2.723000e+03,276.000000,1.765687e+09,1.762017e+09,1.762260e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.759032e+09,1.172470e+06
75%,1.771300e+06,2.122663e+08,7.656120e+16,5.100000e+01,8.000000e+00,1.229500e+04,3.620000e+02,9.001000e+03,1462.000000,1.767210e+09,1.764576e+09,1.764617e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.763067e+09,1.771300e+06
max,3.241660e+06,2.152632e+08,7.656120e+16,3.370000e+04,1.974800e+04,2.457680e+06,3.460800e+04,2.416117e+06,157407.000000,1.767652e+09,1.767650e+09,1.767651e+09,5.763000e+03,1.425000e+03,9.897801e-01,2.580000e+02,1.767650e+09,3.241660e+06


| 한글 컬럼명 | 영문 컬럼명 | 설명(내용) | 범위(실제 분포 기준) |
|---|---|---|---|
| 앱 ID | app_id | Steam 앱 고유 식별자 | 정수 (수천만 단위, 예: 43 ~ 85,000,000+) |
| 앱 이름 | app_name | 게임 또는 앱 이름 | 문자열 |
| 리뷰 ID | review_id | 리뷰 고유 식별자 | 정수 (고유값) |
| 리뷰 언어 | language | 리뷰가 작성된 언어 | 문자열 (예: english, schinese 등) |
| 리뷰 내용 | review | 리뷰 텍스트 본문 | 문자열 (길이 가변) |
| 리뷰 생성 시각 | timestamp_created | 리뷰 작성 시각 | Unix Timestamp (약 1.29B ~ 1.61B) |
| 리뷰 수정 시각 | timestamp_updated | 리뷰 마지막 수정 시각 | Unix Timestamp (약 1.29B ~ 2.28B) |
| 추천 여부 | recommended | 게임 추천 여부 | Boolean (true / false, true ≈ 87%) |
| 도움됨 투표 수 | votes_helpful | 도움됨(Helpful) 투표 수 | 정수 (0 ~ 약 4,900) |
| 재미있음 투표 수 | votes_funny | 재미있음(Funny) 투표 수 | 정수 (0 ~ 약 27,000) |
| 가중 투표 점수 | weighted_vote_score | 도움됨 기반 가중 점수 | 실수 (0.0 ~ 1.0) |
| 댓글 수 | comment_count | 리뷰 댓글 수 | 정수 (0 ~ 약 1,300,000) |
| 스팀 구매 여부 | steam_purchase | Steam에서 직접 구매했는지 여부 | Boolean (true ≈ 77%) |
| 무료 획득 여부 | received_for_free | 무료 획득 여부 | Boolean (true ≈ 3%) |
| 얼리액세스 리뷰 | written_during_early_access | 얼리 액세스 중 작성 여부 | Boolean (true ≈ 9%) |
| 작성자 SteamID | author.steamid | 리뷰 작성자 SteamID | 64-bit 정수 (약 7.6e16 ~ 7.7e16) |
| 작성자 보유 게임 수 | author.num_games_owned | 보유 게임 개수 | 정수 (0 ~ 약 21,700,000) |
| 작성자 리뷰 수 | author.num_reviews | 작성자 전체 리뷰 수 | 정수 (0 ~ 약 1,290,000) |
| 누적 플레이 시간 | author.playtime_forever | 총 플레이 시간 | 초 단위 정수 (0 ~ 약 85,000,000초) |
| 최근 2주 플레이 시간 | author.playtime_last_two_weeks | 최근 2주 플레이 시간 | 초 단위 정수 (0 ~ 약 3,700,000초) |
| 리뷰 시점 플레이 시간 | author.playtime_at_review | 리뷰 당시 플레이 시간 | 초 단위 정수 (0 ~ 약 27,000초) |
| 마지막 플레이 시각 | author.last_played | 마지막 플레이 시각 | Unix Timestamp (약 1.29B ~ 1.61B) |


In [244]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030656 entries, 0 to 1030655
Data columns (total 28 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   appid                        1030656 non-null  int64  
 1   recommendationid             1030656 non-null  int64  
 2   steamid                      1030656 non-null  int64  
 3   num_games_owned              1030656 non-null  int64  
 4   num_reviews_author           1030656 non-null  int64  
 5   playtime_forever             1030645 non-null  float64
 6   playtime_last_two_weeks      1030645 non-null  float64
 7   playtime_at_review           1030656 non-null  int64  
 8   deck_playtime_at_review      19544 non-null    float64
 9   last_played                  1030645 non-null  float64
 10  language                     1030656 non-null  object 
 11  review                       1027088 non-null  object 
 12  timestamp_created            1030656 non-n

In [245]:
df.isnull().sum()

appid                                0
recommendationid                     0
steamid                              0
num_games_owned                      0
num_reviews_author                   0
playtime_forever                    11
playtime_last_two_weeks             11
playtime_at_review                   0
deck_playtime_at_review        1011112
last_played                         11
language                             0
review                            3568
timestamp_created                    0
timestamp_updated                    0
voted_up                             0
votes_up                             0
votes_funny                          0
weighted_vote_score                  0
comment_count                        0
steam_purchase                       0
received_for_free                    0
written_during_early_access          0
developer_response             1027150
timestamp_dev_responded        1027150
primarily_steam_deck                 0
appid_1                  

In [246]:
df.describe()

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,timestamp_created,timestamp_updated,votes_up,votes_funny,weighted_vote_score,comment_count,timestamp_dev_responded,appid_1
count,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030645e+06,1.030645e+06,1.030656e+06,19544.000000,1.030645e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,1.030656e+06,3.506000e+03,1.030656e+06
mean,1.246007e+06,2.078202e+08,7.656120e+16,5.589029e+01,9.407134e+00,1.421312e+04,4.750142e+02,1.179464e+04,1883.487055,1.762456e+09,1.760941e+09,1.761054e+09,6.485481e-01,1.244169e-01,5.027320e-01,4.645294e-02,1.759162e+09,1.246007e+06
std,8.606580e+05,4.897537e+06,6.086190e+08,1.973416e+02,6.927022e+01,3.357524e+04,1.155320e+03,3.145326e+04,5554.090679,1.173887e+07,4.747926e+06,4.725906e+06,1.345458e+01,3.082606e+00,2.275669e-02,6.260744e-01,4.410209e+06,8.606580e+05
min,4.400000e+02,1.994023e+08,7.656120e+16,0.000000e+00,1.000000e+00,5.000000e+00,0.000000e+00,5.000000e+00,1.000000,1.345532e+09,1.752096e+09,1.752096e+09,0.000000e+00,0.000000e+00,1.352486e-01,0.000000e+00,1.752181e+09,4.400000e+02
25%,5.268700e+05,2.032641e+08,7.656120e+16,0.000000e+00,1.000000e+00,1.440000e+03,0.000000e+00,8.080000e+02,46.000000,1.761593e+09,1.756657e+09,1.756828e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.755116e+09,5.268700e+05
50%,1.172470e+06,2.081491e+08,7.656120e+16,0.000000e+00,3.000000e+00,4.408000e+03,0.000000e+00,2.723000e+03,276.000000,1.765687e+09,1.762017e+09,1.762260e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.759032e+09,1.172470e+06
75%,1.771300e+06,2.122663e+08,7.656120e+16,5.100000e+01,8.000000e+00,1.229500e+04,3.620000e+02,9.001000e+03,1462.000000,1.767210e+09,1.764576e+09,1.764617e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.763067e+09,1.771300e+06
max,3.241660e+06,2.152632e+08,7.656120e+16,3.370000e+04,1.974800e+04,2.457680e+06,3.460800e+04,2.416117e+06,157407.000000,1.767652e+09,1.767650e+09,1.767651e+09,5.763000e+03,1.425000e+03,9.897801e-01,2.580000e+02,1.767650e+09,3.241660e+06


In [247]:
df.columns

Index(['appid', 'recommendationid', 'steamid', 'num_games_owned',
       'num_reviews_author', 'playtime_forever', 'playtime_last_two_weeks',
       'playtime_at_review', 'deck_playtime_at_review', 'last_played',
       'language', 'review', 'timestamp_created', 'timestamp_updated',
       'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score',
       'comment_count', 'steam_purchase', 'received_for_free',
       'written_during_early_access', 'developer_response',
       'timestamp_dev_responded', 'primarily_steam_deck', 'appid_1',
       'game_name', 'genre'],
      dtype='object')

In [ ]:
# # 도메인으로 필터링 함
# drop_columns = [
#     'app_id',
#     'app_name',
#     'review_id',
# ]
# df = df.drop(columns=drop_columns)
# df

In [249]:
df.count()

appid                          1030656
recommendationid               1030656
steamid                        1030656
num_games_owned                1030656
num_reviews_author             1030656
playtime_forever               1030645
playtime_last_two_weeks        1030645
playtime_at_review             1030656
deck_playtime_at_review          19544
last_played                    1030645
language                       1030656
review                         1027088
timestamp_created              1030656
timestamp_updated              1030656
voted_up                       1030656
votes_up                       1030656
votes_funny                    1030656
weighted_vote_score            1030656
comment_count                  1030656
steam_purchase                 1030656
received_for_free              1030656
written_during_early_access    1030656
developer_response                3506
timestamp_dev_responded           3506
primarily_steam_deck           1030656
appid_1                  

In [ ]:
review_dt = pd.to_datetime(df["timestamp_created"], unit="s")
last_dt   = pd.to_datetime(df["last_played"], unit="s")

df["days_after_review"] = (last_dt - review_dt).dt.days

df["churn"] = (df["days_after_review"] < 30).astype(int)

# 예외 처리
df.loc[df["last_played"] == 0, "churn"] = 1
df.loc[df["days_after_review"] < 0, "churn"] = 1


In [251]:
# df['days_since_last_play']


In [252]:
df[df["churn"]==1].count() 

appid                          621559
recommendationid               621559
steamid                        621559
num_games_owned                621559
num_reviews_author             621559
playtime_forever               621559
playtime_last_two_weeks        621559
playtime_at_review             621559
deck_playtime_at_review         12448
last_played                    621559
language                       621559
review                         619418
timestamp_created              621559
timestamp_updated              621559
voted_up                       621559
votes_up                       621559
votes_funny                    621559
weighted_vote_score            621559
comment_count                  621559
steam_purchase                 621559
received_for_free              621559
written_during_early_access    621559
developer_response               1587
timestamp_dev_responded          1587
primarily_steam_deck           621559
appid_1                        621559
game_name   

In [253]:
df

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,developer_response,timestamp_dev_responded,primarily_steam_deck,appid_1,game_name,genre,days_after_review,churn
0,2139460,199685023,76561198220582271,23,1,15240.0,0.0,15240,NaN,1.748034e+09,turkish,oyunu kurduk eyv oynadık vs zaten türk ü bırak oyuncu bulmak çok zor sadece rus kekolar var bol bol hadi bunları geçtik diyelim oynadık durduk bir güncelleme geldi neymiş yeni senaryo imiş tamam eyv süper güncelleme geliyor bi baktım 70 gb dedik oyun bayağı değişiyor gelişiyor neyse yaptık güncellemeyi baktım bi tek senaryo gelmiş değişen birşey yok tamam dedik eyv. geçen bir kere daha gireyim dedim yeni senaryo gelmiş bi baktım şimdi 64 gb lık bir güncelleme daha .Olum siz manyak mısınız zaten tek tük oyuncu var oynayan birde her senaryoda 70gb güncelleme mi olur mk. mal mısınız sildim sokarım oyununuza. Undawn a devam mk,1752389967,1752389967,False,17,2,0.712445,1,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']",-51.0,1
1,2139460,199684668,76561198401253542,0,1,6249.0,0.0,834,NaN,1.755746e+09,spanish,me esta fasinando,1752389691,1752389691,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']",38.0,0
2,2139460,199683985,76561199435851437,0,1,20522.0,0.0,12327,NaN,1.755124e+09,english,truly an amazing game,1752389204,1752389204,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']",31.0,0
3,2139460,199683578,76561198977144059,83,30,22726.0,0.0,4090,NaN,1.755366e+09,english,amazing free to play .,1752388914,1752388914,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']",34.0,0
4,2139460,199683575,76561199109403538,0,1,2747.0,0.0,2389,NaN,1.752985e+09,schinese,好,1752388911,1752388911,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']",6.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1030651,526870,215146158,76561198022734436,95,4,13884.0,645.0,13238,NaN,1.767635e+09,english,Addicting if you like factory games,1767549535,1767549535,True,0,0,0.500000,0,False,False,False,NaN,NaN,False,526870,Satisfactory,"['Adventure', 'Indie', 'Simulation', 'Strategy']",0.0,1
1030652,526870,215145064,76561199525731410,0,3,4075.0,4075.0,3547,NaN,1.767588e+09,english,very fun,1767548829,1767548829,True,0,0,0.500000,0,True,False,False,NaN,NaN,False,526870,Satisfactory,"['Adventure', 'Indie', 'Simulation', 'Strategy']",0.0,1
1030653,526870,215144030,76561199864838845,0,3,467.0,467.0,364,NaN,1.767635e+09,english,fun,1767548108,1767548108,True,0,0,0.500000,0,True,False,False,NaN,NaN,False,526870,Satisfactory,"['Adventure', 'Indie', 'Simulation', 'Strategy']",1.0,1
1030654,526870,215143393,76561199698303694,0,1,8166.0,946.0,8166,NaN,1.767463e+09,norwegian,"136 houres into the game, love it. Don,t play this gmae is you want to keep you job.",1767547704,1767547704,True,0,0,0.500000,0,True,False,False,NaN,NaN,False,526870,Satisfactory,"['Adventure', 'Indie', 'Simulation', 'Strategy']",-1.0,1


In [254]:
df['genre']

0          ['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']
1          ['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']
2          ['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']
3          ['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']
4          ['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']
                                             ...                                   
1030651                            ['Adventure', 'Indie', 'Simulation', 'Strategy']
1030652                            ['Adventure', 'Indie', 'Simulation', 'Strategy']
1030653                            ['Adventure', 'Indie', 'Simulation', 'Strategy']
1030654                            ['Adventure', 'Indie', 'Simulation', 'Strategy']
1030655                            ['Adventure', 'Indie', 'Simulation', 'Strategy']
Name: genre, Length: 1030656, dtype: object

In [263]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# =========================
# 1) datetime + 기본 전처리
# =========================
df = df.copy()

df["review_dt"] = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
df = df.dropna(subset=["review_dt"]).copy()

# review_length
df["review_length"] = df["review"].fillna("").astype(str).str.len()

# deck_playtime_at_review 결측 처리 (컬럼 있으면)
if "deck_playtime_at_review" in df.columns:
    df["deck_playtime_at_review"] = df["deck_playtime_at_review"].fillna(0)

# True/False -> 0/1 정리 (LightGBM 편하게)
bool_cols = [
    "primarily_steam_deck",
    "voted_up",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
]
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].astype(int)


ending_genre = [
  "Visual Novel",
  "Interactive Fiction",
  "Walking Simulator",
  "Story Rich",
  "Adventure",
  "Puzzle",
  "Horror",
  "Mystery",
  "Psychological Horror",
  "Narrative"
]

df["is_ending_genre"] = df["genre"].apply(
    lambda g: int(any(x in g for x in ending_genre))
)

# =========================
# 2) 180일 중 마지막 90일 제외 (라벨 생성용 구간만)
# =========================
END_DATE = df["review_dt"].max()
START_DATE = END_DATE - pd.Timedelta(days=180)
LABEL_CUTOFF = END_DATE - pd.Timedelta(days=60)

df_180 = df[df["review_dt"] >= START_DATE].copy()
df_label = df_180[df_180["review_dt"] <= LABEL_CUTOFF].copy()

# =========================
# 3) churn 라벨 (프록시) 생성
# - last_played는 피처로 쓰지 않지만, 라벨 생성엔 사용
# - 리뷰 이후 90일 안에 last_played가 "없거나/그 이전이면" churn=1 로 가정
# =========================
df_label["last_played_dt"] = pd.to_datetime(df_label["last_played"], unit="s", errors="coerce")

# last_played가 NaT면 복귀 관측 안됨 -> churn=1 처리
df_label["churn"] = (
    df_label["last_played_dt"].isna()
    | (df_label["last_played_dt"] <= (df_label["review_dt"] + pd.Timedelta(days=30)))
).astype(int)

# =========================
# 4) 피처 선택 (사용자가 준 리스트 그대로)
# =========================
features = [
    "num_games_owned",
    "num_reviews_author",
    "deck_playtime_at_review",
    "voted_up",
    "votes_up",
    "votes_funny",
    "weighted_vote_score",
    "comment_count",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "review_length",
    "is_ending_genre"
]

# 존재하는 컬럼만 사용 (실행 에러 방지)
features = [c for c in features if c in df_label.columns]


# 숫자형 강제 (문자 섞이면 터짐)
for c in features:
    df_label[c] = pd.to_numeric(df_label[c], errors="coerce")

# 결측은 0으로 (간단 버전). 더 정교하게 하려면 median 등으로 대체.
X = df_label[features].fillna(0)
y = df_label["churn"].astype(int)

# =========================
# 5) 시간 기준 Train/Valid Split (마지막 30일을 valid)
# =========================
split_date = LABEL_CUTOFF - pd.Timedelta(days=30)

train_mask = df_label["review_dt"] <= split_date
valid_mask = df_label["review_dt"] > split_date

X_train, y_train = X[train_mask], y[train_mask]
X_valid, y_valid = X[valid_mask], y[valid_mask]

print("Rows:", len(df_label), "| Train:", len(X_train), "| Valid:", len(X_valid))
print("Churn rate train:", round(y_train.mean(), 4), "| valid:", round(y_valid.mean(), 4))
print("Features used:", features)

# valid에 한 클래스만 있으면 AUC 계산이 안 됨
if y_valid.nunique() < 2:
    raise ValueError(f"Valid set에 클래스가 1개뿐입니다. (unique={y_valid.unique()}) split_date를 조정하거나 기간을 늘려야 합니다.")

# =========================
# 6) LightGBM 학습 + ROC-AUC
# =========================
model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc"
)

pred = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, pred)
print(f"Validation ROC-AUC: {auc:.4f}")

# =========================
# 7) (선택) 최근 90일(inference 구간) churn 확률 예측
# =========================
df_recent = df_180[df_180["review_dt"] > LABEL_CUTOFF].copy()

# 최근구간에도 동일 전처리 적용
df_recent["review_length"] = df_recent["review"].fillna("").astype(str).str.len()
if "deck_playtime_at_review" in df_recent.columns:
    df_recent["deck_playtime_at_review"] = df_recent["deck_playtime_at_review"].fillna(0)
for c in bool_cols:
    if c in df_recent.columns:
        df_recent[c] = df_recent[c].astype(int)
for c in features:
    df_recent[c] = pd.to_numeric(df_recent[c], errors="coerce")

X_recent = df_recent[features].fillna(0)
df_recent["churn_prob"] = model.predict_proba(X_recent)[:, 1]

df_recent[["steamid", "appid", "review_dt", "churn_prob"]].head()


Rows: 535073 | Train: 414523 | Valid: 120550
Churn rate train: 0.3508 | valid: 0.4337
Features used: ['num_games_owned', 'num_reviews_author', 'deck_playtime_at_review', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'review_length', 'is_ending_genre']
[LightGBM] [Info] Number of positive: 145418, number of negative: 269105
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010307 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1517
[LightGBM] [Info] Number of data points in the train set: 414523, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.350808 -> initscore=-0.615489
[LightGBM] [Info] Start training from score -0.615489
Validation ROC-AUC: 0.6398


,steamid,appid,review_dt,churn_prob
655,76561198179548413,960170,2026-01-05 19:14:12,0.267486
656,76561198441860447,960170,2026-01-05 15:45:25,0.397021
657,76561199146670949,960170,2026-01-05 14:07:02,0.237051
658,76561198990624310,960170,2026-01-05 13:24:47,0.332537
659,76561198381171927,960170,2026-01-05 13:21:07,0.469171


In [256]:
df[df['is_ending_genre']==0].count()

appid                          503651
recommendationid               503651
steamid                        503651
num_games_owned                503651
num_reviews_author             503651
playtime_forever               503646
playtime_last_two_weeks        503646
playtime_at_review             503651
deck_playtime_at_review        503651
last_played                    503646
language                       503651
review                         502044
timestamp_created              503651
timestamp_updated              503651
voted_up                       503651
votes_up                       503651
votes_funny                    503651
weighted_vote_score            503651
comment_count                  503651
steam_purchase                 503651
received_for_free              503651
written_during_early_access    503651
developer_response                442
timestamp_dev_responded           442
primarily_steam_deck           503651
appid_1                        503651
game_name   

In [257]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix, f1_score, balanced_accuracy_score
)

def sweep_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 19)
    rows = []

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        rows.append({
            "threshold": round(float(t), 2),
            "precision_1": tp / (tp + fp + 1e-9),
            "recall_1": tp / (tp + fn + 1e-9),
            "precision_0": tn / (tn + fn + 1e-9),
            "recall_0": tn / (tn + fp + 1e-9),
            "f1_macro": f1_score(y_true, y_pred, average="macro"),
            "balanced_acc": balanced_accuracy_score(y_true, y_pred),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn
        })

    return rows

rows = sweep_threshold(y_valid, y_prob)
for r in rows:
    print(r)

{'threshold': 0.05, 'precision_1': np.float64(0.4337453338863506), 'recall_1': np.float64(0.9999999999999809), 'precision_0': np.float64(0.0), 'recall_0': np.float64(0.0), 'f1_macro': 0.3025260648699939, 'balanced_acc': 0.5, 'tp': np.int64(52288), 'fp': np.int64(68262), 'fn': np.int64(0), 'tn': np.int64(0)}
{'threshold': 0.1, 'precision_1': np.float64(0.4337453338863506), 'recall_1': np.float64(0.9999999999999809), 'precision_0': np.float64(0.0), 'recall_0': np.float64(0.0), 'f1_macro': 0.3025260648699939, 'balanced_acc': 0.5, 'tp': np.int64(52288), 'fp': np.int64(68262), 'fn': np.int64(0), 'tn': np.int64(0)}
{'threshold': 0.15, 'precision_1': np.float64(0.4337453338863506), 'recall_1': np.float64(0.9999999999999809), 'precision_0': np.float64(0.0), 'recall_0': np.float64(0.0), 'f1_macro': 0.3025260648699939, 'balanced_acc': 0.5, 'tp': np.int64(52288), 'fp': np.int64(68262), 'fn': np.int64(0), 'tn': np.int64(0)}
{'threshold': 0.2, 'precision_1': np.float64(0.4337453338863506), 'recall_

In [261]:
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)

def train_eval_linear_svm(
    X_train, y_train,
    X_valid, y_valid,
    C=1.0,
    calibrate=True,
    calib_method="sigmoid",
    calib_cv=3,
    threshold=0.3
):
    base = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("svm", LinearSVC(C=C, class_weight="balanced", random_state=42))
    ])

    if calibrate:
        clf = CalibratedClassifierCV(base, method=calib_method, cv=calib_cv)
    else:
        clf = base

    clf.fit(X_train, y_train)

    # ---- 확률 / 점수 ----
    if calibrate:
        y_prob = clf.predict_proba(X_valid)[:, 1]
    else:
        y_prob = clf.decision_function(X_valid)

    # ---- 지표 ----
    auc = roc_auc_score(y_valid, y_prob)
    pr_auc = average_precision_score(y_valid, y_prob)

    if calibrate:
        y_pred = (y_prob >= threshold).astype(int)
    else:
        y_pred = (y_prob >= 0).astype(int)

    print(f"[Linear SVM] C={C} | calibrate={calibrate} ({calib_method})")
    print(f"ROC-AUC: {auc:.4f}")
    print(f"PR-AUC : {pr_auc:.4f}\n")
    print("Confusion matrix:")
    print(confusion_matrix(y_valid, y_pred), "\n")
    print("Classification report:")
    print(classification_report(y_valid, y_pred, digits=4))

    # 🔥 y_prob 같이 반환
    return clf, y_prob, {"roc_auc": auc, "pr_auc": pr_auc}



#===== 사용 예시 =====
#X_train, y_train, X_valid, y_valid 가 이미 준비돼 있다고 가정
#(너는 LightGBM 돌렸던 그대로 같은 split을 쓰는 게 핵심)

#1) 기본 1회 실행
clf, y_prob, metrics = train_eval_linear_svm(
    X_train, y_train,
    X_valid, y_valid,
    C=1.0,
    calibrate=True
)
#2) C 몇 개만 간단 스윕
for C in [0.01, 0.1, 1.0, 5.0, 10.0]:
    _clf, _prob , _m = train_eval_linear_svm(X_train, y_train, X_valid, y_valid, C=C, calibrate=True)


[Linear SVM] C=1.0 | calibrate=True (sigmoid)
ROC-AUC: 0.5998
PR-AUC : 0.5396

Confusion matrix:
[[17749 50513]
 [ 8858 43430]] 

Classification report:
              precision    recall  f1-score   support

           0     0.6671    0.2600    0.3742     68262
           1     0.4623    0.8306    0.5940     52288

    accuracy                         0.5075    120550
   macro avg     0.5647    0.5453    0.4841    120550
weighted avg     0.5783    0.5075    0.4695    120550

[Linear SVM] C=0.01 | calibrate=True (sigmoid)
ROC-AUC: 0.6000
PR-AUC : 0.5396

Confusion matrix:
[[17849 50413]
 [ 8899 43389]] 

Classification report:
              precision    recall  f1-score   support

           0     0.6673    0.2615    0.3757     68262
           1     0.4626    0.8298    0.5940     52288

    accuracy                         0.5080    120550
   macro avg     0.5649    0.5456    0.4849    120550
weighted avg     0.5785    0.5080    0.4704    120550

[Linear SVM] C=0.1 | calibrate=True (si

In [259]:
from sklearn.metrics import f1_score, balanced_accuracy_score

def sweep_thresholds(y_true, y_prob, thresholds=np.linspace(0.01, 0.99, 99)):
    best = None
    rows = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        f1m = f1_score(y_true, y_pred, average="macro")
        rows.append((t, bal_acc, f1m, tp, fp, fn, tn))
    rows = sorted(rows, key=lambda x: x[1], reverse=True)  # bal_acc 기준
    return rows[:10]

top10 = sweep_thresholds(y_valid, y_prob)
top10

[(np.float64(0.34),
  0.5720226844320431,
  0.5702678242211018,
  np.int64(29312),
  np.int64(28434),
  np.int64(22976),
  np.int64(39828)),
 (np.float64(0.36000000000000004),
  0.5709828042864187,
  0.5639130290583312,
  np.int64(19191),
  np.int64(15363),
  np.int64(33097),
  np.int64(52899)),
 (np.float64(0.35000000000000003),
  0.5702440358382117,
  0.5697887282073633,
  np.int64(23689),
  np.int64(21336),
  np.int64(28599),
  np.int64(46926)),
 (np.float64(0.37),
  0.5643815336699334,
  0.549737820499498,
  np.int64(16349),
  np.int64(12554),
  np.int64(35939),
  np.int64(55708)),
 (np.float64(0.38),
  0.5582007037437152,
  0.5362983671379336,
  np.int64(14307),
  np.int64(10732),
  np.int64(37981),
  np.int64(57530)),
 (np.float64(0.39),
  0.5535265767920002,
  0.5243144785843543,
  np.int64(12647),
  np.int64(9203),
  np.int64(39641),
  np.int64(59059)),
 (np.float64(0.33),
  0.5506692567932604,
  0.522455000481298,
  np.int64(37956),
  np.int64(42634),
  np.int64(14332),
  np.i

In [260]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix, f1_score, balanced_accuracy_score
)

def sweep_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 19)
    rows = []

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        rows.append({
            "threshold": round(float(t), 2),
            "precision_1": tp / (tp + fp + 1e-9),
            "recall_1": tp / (tp + fn + 1e-9),
            "precision_0": tn / (tn + fn + 1e-9),
            "recall_0": tn / (tn + fp + 1e-9),
            "f1_macro": f1_score(y_true, y_pred, average="macro"),
            "balanced_acc": balanced_accuracy_score(y_true, y_pred),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn
        })

    return rows

rows = sweep_threshold(y_valid, y_prob)
for r in rows:
    print(r)

{'threshold': 0.05, 'precision_1': np.float64(0.4337453338863506), 'recall_1': np.float64(0.9999999999999809), 'precision_0': np.float64(0.0), 'recall_0': np.float64(0.0), 'f1_macro': 0.3025260648699939, 'balanced_acc': 0.5, 'tp': np.int64(52288), 'fp': np.int64(68262), 'fn': np.int64(0), 'tn': np.int64(0)}
{'threshold': 0.1, 'precision_1': np.float64(0.4337525301124827), 'recall_1': np.float64(0.9999999999999809), 'precision_0': np.float64(0.9999999995), 'recall_0': np.float64(2.92988778529778e-05), 'f1_macro': 0.3025588636192132, 'balanced_acc': 0.5000146494389265, 'tp': np.int64(52288), 'fp': np.int64(68260), 'fn': np.int64(0), 'tn': np.int64(2)}
{'threshold': 0.15, 'precision_1': np.float64(0.4337597265774025), 'recall_1': np.float64(0.9999999999999809), 'precision_0': np.float64(0.99999999975), 'recall_0': np.float64(5.85977557059556e-05), 'f1_macro': 0.3025916607327544, 'balanced_acc': 0.500029298877853, 'tp': np.int64(52288), 'fp': np.int64(68258), 'fn': np.int64(0), 'tn': np.in